In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import pickle
import os

print("All imports successful")

All imports successful


In [2]:
# Load dataset
df = pd.read_csv("../data/train.csv")

# Keep only what we need
df = df[["comment_text", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]].dropna()

print("Shape:", df.shape)
print("Toxic rate:", round(df["toxic"].mean() * 100, 2), "%")
print("Sample comment:")
print(df["comment_text"].iloc[0][:200])

Shape: (159571, 7)
Toxic rate: 9.58 %
Sample comment:
Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove th


In [3]:
X = df["comment_text"]
y = df["toxic"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Toxic in test:", y_test.sum(), f"({round(y_test.mean()*100,2)}%)")

Train size: 127656
Test size: 31915
Toxic in test: 3059 (9.58%)


In [4]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train matrix shape:", X_train_tfidf.shape)
print("Test matrix shape:", X_test_tfidf.shape)

Train matrix shape: (127656, 20000)
Test matrix shape: (31915, 20000)


In [5]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))

              precision    recall  f1-score   support

           0       0.98      0.95      0.97     28856
           1       0.64      0.86      0.73      3059

    accuracy                           0.94     31915
   macro avg       0.81      0.90      0.85     31915
weighted avg       0.95      0.94      0.94     31915

F1 Score: 0.7326


In [6]:
# Get probability scores instead of just 0/1 predictions
ml_prob = model.predict_proba(X_test_tfidf)[:, 1]

# Build evaluation dataframe
df_eval = pd.DataFrame({
    "comment_text": X_test.reset_index(drop=True),
    "true_label": y_test.reset_index(drop=True),
    "ml_prob": ml_prob
})

print("Eval dataframe shape:", df_eval.shape)
print("\nSample:")
print(df_eval.head(3))

Eval dataframe shape: (31915, 3)

Sample:
                                        comment_text  true_label   ml_prob
0  "\n""oo9"", if you have ANY actual evidence, s...           0  0.665178
1  It seems that AE defend anything - including v...           0  0.389875
2  Don Patch's Parents and Childhood ==\n\nI thin...           0  0.111990


In [7]:
def ml_bucket(prob):
    if prob >= 0.8: return "HIGH"
    if prob >= 0.5: return "MED"
    if prob >= 0.2: return "LOW"
    return "SAFE"

def action_from_bucket(bucket):
    if bucket == "HIGH": return "AUTO_BLOCK"
    if bucket == "MED": return "HUMAN_REVIEW"
    if bucket == "LOW": return "DOWNRANK"
    return "ALLOW"

df_eval["ml_bucket"] = df_eval["ml_prob"].apply(ml_bucket)
df_eval["ml_action"] = df_eval["ml_bucket"].apply(action_from_bucket)

print("Risk bucket distribution:")
print(df_eval["ml_bucket"].value_counts())
print()
print("Action distribution:")
print(df_eval["ml_action"].value_counts())

Risk bucket distribution:
ml_bucket
SAFE    23013
LOW      4767
HIGH     2517
MED      1618
Name: count, dtype: int64

Action distribution:
ml_action
ALLOW           23013
DOWNRANK         4767
AUTO_BLOCK       2517
HUMAN_REVIEW     1618
Name: count, dtype: int64


In [8]:
import os
os.makedirs("../models", exist_ok=True)

# Save model
with open("../models/lr_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("../models/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save eval dataframe
df_eval.to_csv("../models/df_eval.csv", index=False)

print("Model saved ✓")
print("Vectorizer saved ✓")
print("Eval dataframe saved ✓")

Model saved ✓
Vectorizer saved ✓
Eval dataframe saved ✓
